**Steps**

1. Load the processed structured data.
2. Separate development and held-out studies.
3. Prepare numerical structured features.
4. Train a simple MLP classifier.
5. Evaluate the structured-only baseline.
6. Extract structured feature representations.
7. Save the trained MLP and feature vectors.

### Main output

`structured_features_mlp.npy`

In [1]:
import os
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import (
    TensorDataset,
    DataLoader
)

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
base_path = (
    '/content/drive/MyDrive/'
    'dissertation_project/data'
)

processed_path = (
    f'{base_path}/processed'
)

model_path = (
    f'{base_path}/models'
)

os.makedirs(
    model_path,
    exist_ok=True
)

print(
    "Processed path:",
    processed_path
)

print(
    "Model path:",
    model_path
)

Processed path: /content/drive/MyDrive/dissertation_project/data/processed
Model path: /content/drive/MyDrive/dissertation_project/data/models


In [3]:
device = torch.device(
    'cuda'
    if torch.cuda.is_available()
    else 'cpu'
)

print(
    "Device:",
    device
)

Device: cpu


In [4]:
# Load structured data
structured_file = (
    f'{processed_path}/'
    'structured_processed.csv'
)

structured_data = pd.read_csv(
    structured_file
)

print(
    "Structured data shape:",
    structured_data.shape
)

print(
    "\nColumns:"
)

print(
    structured_data.columns.tolist()
)

Structured data shape: (2200, 66)

Columns:
['subject_id', 'study_id', 'hadm_id', 'anchor_age', 'length_of_stay_hours', 'glucose', 'creatinine', 'sodium', 'potassium', 'hemoglobin', 'platelet_count', 'urea_nitrogen', 'No Finding', 'Support Devices', 'Pleural Effusion', 'Lung Opacity', 'Atelectasis', 'Cardiomegaly', 'Edema', 'gender_M', 'admission_type_DIRECT EMER.', 'admission_type_DIRECT OBSERVATION', 'admission_type_ELECTIVE', 'admission_type_EU OBSERVATION', 'admission_type_EW EMER.', 'admission_type_OBSERVATION ADMIT', 'admission_type_SURGICAL SAME DAY ADMISSION', 'admission_type_URGENT', 'insurance_Medicare', 'insurance_Other', 'insurance_Private', 'insurance_Unknown', 'marital_status_MARRIED', 'marital_status_SINGLE', 'marital_status_Unknown', 'marital_status_WIDOWED', 'race_ASIAN', 'race_ASIAN - ASIAN INDIAN', 'race_ASIAN - CHINESE', 'race_ASIAN - KOREAN', 'race_ASIAN - SOUTH EAST ASIAN', 'race_BLACK/AFRICAN', 'race_BLACK/AFRICAN AMERICAN', 'race_BLACK/CAPE VERDEAN', 'race_BLACK

In [5]:
# Study IDs
structured_data['study_id'] = (
    structured_data['study_id']
    .astype(str)
)

print(
    "Unique studies:",
    structured_data[
        'study_id'
    ].nunique()
)

Unique studies: 2200


In [6]:
# Define target labels
label_columns = [
    'No Finding',
    'Support Devices',
    'Pleural Effusion',
    'Lung Opacity',
    'Atelectasis',
    'Cardiomegaly',
    'Edema'
]

available_labels = [
    label
    for label in label_columns
    if label in structured_data.columns
]

print(
    "Target labels:"
)

print(
    available_labels
)

Target labels:
['No Finding', 'Support Devices', 'Pleural Effusion', 'Lung Opacity', 'Atelectasis', 'Cardiomegaly', 'Edema']


In [7]:
# Convert targets to binary
for label in available_labels:

    structured_data[label] = pd.to_numeric(
        structured_data[label],
        errors='coerce'
    ).fillna(0)

    structured_data[label] = (
        structured_data[label] == 1
    ).astype(np.float32)


print(
    "Target values:"
)

for label in available_labels:

    print(
        label,
        sorted(
            structured_data[
                label
            ].unique().tolist()
        )
    )

Target values:
No Finding [0.0, 1.0]
Support Devices [0.0, 1.0]
Pleural Effusion [0.0, 1.0]
Lung Opacity [0.0, 1.0]
Atelectasis [0.0, 1.0]
Cardiomegaly [0.0, 1.0]
Edema [0.0, 1.0]


In [8]:
# Load held-out IDs
heldout_ids_file = (
    f'{processed_path}/'
    'test_calibration_ids.csv'
)

heldout_ids = pd.read_csv(
    heldout_ids_file
)

heldout_ids['study_id'] = (
    heldout_ids['study_id']
    .astype(str)
)

heldout_id_set = set(
    heldout_ids['study_id']
)

print(
    "Held-out studies:",
    len(heldout_id_set)
)

Held-out studies: 687


In [9]:
# Identify structured features
excluded_columns = (
    ['study_id']
    + available_labels
)

feature_columns = [
    column
    for column in structured_data.columns
    if column not in excluded_columns
]

print(
    "Number of structured features:",
    len(feature_columns)
)

print(
    "\nStructured features:"
)

print(
    feature_columns
)

Number of structured features: 58

Structured features:
['subject_id', 'hadm_id', 'anchor_age', 'length_of_stay_hours', 'glucose', 'creatinine', 'sodium', 'potassium', 'hemoglobin', 'platelet_count', 'urea_nitrogen', 'gender_M', 'admission_type_DIRECT EMER.', 'admission_type_DIRECT OBSERVATION', 'admission_type_ELECTIVE', 'admission_type_EU OBSERVATION', 'admission_type_EW EMER.', 'admission_type_OBSERVATION ADMIT', 'admission_type_SURGICAL SAME DAY ADMISSION', 'admission_type_URGENT', 'insurance_Medicare', 'insurance_Other', 'insurance_Private', 'insurance_Unknown', 'marital_status_MARRIED', 'marital_status_SINGLE', 'marital_status_Unknown', 'marital_status_WIDOWED', 'race_ASIAN', 'race_ASIAN - ASIAN INDIAN', 'race_ASIAN - CHINESE', 'race_ASIAN - KOREAN', 'race_ASIAN - SOUTH EAST ASIAN', 'race_BLACK/AFRICAN', 'race_BLACK/AFRICAN AMERICAN', 'race_BLACK/CAPE VERDEAN', 'race_BLACK/CARIBBEAN ISLAND', 'race_HISPANIC OR LATINO', 'race_HISPANIC/LATINO - CENTRAL AMERICAN', 'race_HISPANIC/LATI

In [10]:
# Convert structured features to numeric
for column in feature_columns:

    structured_data[column] = pd.to_numeric(
        structured_data[column],
        errors='coerce'
    )

print(
    structured_data[
        feature_columns
    ].dtypes
)

subject_id                                          int64
hadm_id                                             int64
anchor_age                                        float64
length_of_stay_hours                              float64
glucose                                           float64
creatinine                                        float64
sodium                                            float64
potassium                                         float64
hemoglobin                                        float64
platelet_count                                    float64
urea_nitrogen                                     float64
gender_M                                          float64
admission_type_DIRECT EMER.                       float64
admission_type_DIRECT OBSERVATION                 float64
admission_type_ELECTIVE                           float64
admission_type_EU OBSERVATION                     float64
admission_type_EW EMER.                           float64
admission_type

In [11]:
# Handle missing values
structured_data[
    feature_columns
] = structured_data[
    feature_columns
].fillna(0)

print(
    "Missing values remaining:",
    structured_data[
        feature_columns
    ].isna().sum().sum()
)

Missing values remaining: 0


In [12]:
# Separate development and held-out data
structured_data['is_heldout'] = (
    structured_data['study_id']
    .isin(heldout_id_set)
)

development_data = structured_data[
    ~structured_data['is_heldout']
].copy()

heldout_data = structured_data[
    structured_data['is_heldout']
].copy()

development_data = (
    development_data
    .reset_index(drop=True)
)

heldout_data = (
    heldout_data
    .reset_index(drop=True)
)

print(
    "Development studies:",
    len(development_data)
)

print(
    "Held-out studies:",
    len(heldout_data)
)

Development studies: 1513
Held-out studies: 687


In [13]:
# Create X and y
X_development = (
    development_data[
        feature_columns
    ]
    .values
    .astype(np.float32)
)

y_development = (
    development_data[
        available_labels
    ]
    .values
    .astype(np.float32)
)

X_heldout = (
    heldout_data[
        feature_columns
    ]
    .values
    .astype(np.float32)
)

y_heldout = (
    heldout_data[
        available_labels
    ]
    .values
    .astype(np.float32)
)

print(
    "Development X:",
    X_development.shape
)

print(
    "Development y:",
    y_development.shape
)

print(
    "Held-out X:",
    X_heldout.shape
)

print(
    "Held-out y:",
    y_heldout.shape
)

Development X: (1513, 58)
Development y: (1513, 7)
Held-out X: (687, 58)
Held-out y: (687, 7)


In [14]:
# Standardise structured features
scaler = StandardScaler()

X_development_scaled = (
    scaler.fit_transform(
        X_development
    )
    .astype(np.float32)
)

X_heldout_scaled = (
    scaler.transform(
        X_heldout
    )
    .astype(np.float32)
)

print(
    "Scaled development:",
    X_development_scaled.shape
)

print(
    "Scaled held-out:",
    X_heldout_scaled.shape
)

Scaled development: (1513, 58)
Scaled held-out: (687, 58)


In [15]:
# Internal training/validation split
from sklearn.model_selection import train_test_split

train_idx, validation_idx = (
    train_test_split(
        np.arange(
            len(X_development_scaled)
        ),
        test_size=0.15,
        random_state=42
    )
)

X_train = (
    X_development_scaled[
        train_idx
    ]
)

X_validation = (
    X_development_scaled[
        validation_idx
    ]
)

y_train = (
    y_development[
        train_idx
    ]
)

y_validation = (
    y_development[
        validation_idx
    ]
)

print(
    "Training:",
    X_train.shape
)

print(
    "Validation:",
    X_validation.shape
)

Training: (1286, 58)
Validation: (227, 58)


In [16]:
# Convert to tensors
X_train_tensor = torch.tensor(
    X_train,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.float32
)

X_validation_tensor = torch.tensor(
    X_validation,
    dtype=torch.float32
)

y_validation_tensor = torch.tensor(
    y_validation,
    dtype=torch.float32
)

print(
    X_train_tensor.shape
)

print(
    y_train_tensor.shape
)

torch.Size([1286, 58])
torch.Size([1286, 7])


In [17]:
# DataLoaders
BATCH_SIZE = 32

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

validation_dataset = TensorDataset(
    X_validation_tensor,
    y_validation_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print(
    "Training batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(validation_loader)
)

Training batches: 41
Validation batches: 8


In [18]:
# Define MLP
class StructuredMLP(nn.Module):

    def __init__(
        self,
        input_size,
        num_labels,
        feature_size=128
    ):

        super().__init__()

        self.feature_layer = nn.Sequential(
            nn.Linear(
                input_size,
                128
            ),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        self.classifier = nn.Linear(
            128,
            num_labels
        )

    def forward(
        self,
        x
    ):

        features = (
            self.feature_layer(x)
        )

        logits = (
            self.classifier(features)
        )

        return logits, features


print(
    "Structured MLP class created."
)

Structured MLP class created.


In [19]:
# Create model
mlp_model = StructuredMLP(
    input_size=len(feature_columns),
    num_labels=len(available_labels)
)

mlp_model = mlp_model.to(
    device
)

print(
    mlp_model
)

StructuredMLP(
  (feature_layer): Sequential(
    (0): Linear(in_features=58, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
  )
  (classifier): Linear(in_features=128, out_features=7, bias=True)
)


In [20]:
# Class weights
positive_counts = (
    y_train.sum(axis=0)
)

negative_counts = (
    len(y_train)
    - positive_counts
)

pos_weights = (
    negative_counts
    /
    np.maximum(
        positive_counts,
        1
    )
)

pos_weights = torch.tensor(
    pos_weights,
    dtype=torch.float32
).to(device)

print(
    pd.DataFrame({
        'label': available_labels,
        'positive': positive_counts,
        'negative': negative_counts,
        'weight':
            pos_weights.cpu().numpy()
    })
)

              label  positive  negative    weight
0        No Finding     342.0     944.0  2.760234
1   Support Devices     472.0     814.0  1.724576
2  Pleural Effusion     385.0     901.0  2.340260
3      Lung Opacity     352.0     934.0  2.653409
4       Atelectasis     314.0     972.0  3.095541
5      Cardiomegaly     339.0     947.0  2.793510
6             Edema     185.0    1101.0  5.951351


In [21]:
# Loss and optimizer
criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weights
)

optimizer = torch.optim.Adam(
    mlp_model.parameters(),
    lr=0.001
)

print(
    "MLP optimizer ready."
)

MLP optimizer ready.


In [22]:
# Training function
def train_mlp_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device
):

    model.train()

    total_loss = 0

    for X_batch, y_batch in loader:

        X_batch = X_batch.to(
            device
        )

        y_batch = y_batch.to(
            device
        )

        optimizer.zero_grad()

        logits, _ = model(
            X_batch
        )

        loss = criterion(
            logits,
            y_batch
        )

        loss.backward()

        optimizer.step()

        total_loss += (
            loss.item()
            * X_batch.size(0)
        )

    return (
        total_loss
        /
        len(loader.dataset)
    )

In [23]:
# Validation function
def evaluate_mlp(
    model,
    loader,
    criterion,
    device
):

    model.eval()

    total_loss = 0

    all_labels = []
    all_probabilities = []

    with torch.no_grad():

        for X_batch, y_batch in loader:

            X_batch = X_batch.to(
                device
            )

            y_batch = y_batch.to(
                device
            )

            logits, _ = model(
                X_batch
            )

            probabilities = (
                torch.sigmoid(logits)
            )

            loss = criterion(
                logits,
                y_batch
            )

            total_loss += (
                loss.item()
                * X_batch.size(0)
            )

            all_labels.append(
                y_batch.cpu().numpy()
            )

            all_probabilities.append(
                probabilities.cpu().numpy()
            )

    y_true = np.vstack(
        all_labels
    )

    y_prob = np.vstack(
        all_probabilities
    )

    y_pred = (
        y_prob >= 0.5
    ).astype(int)

    return (
        total_loss / len(loader.dataset),
        y_true,
        y_prob,
        y_pred
    )

In [24]:
# Train MLP
EPOCHS = 10

history = []

for epoch in range(EPOCHS):

    train_loss = train_mlp_epoch(
        mlp_model,
        train_loader,
        optimizer,
        criterion,
        device
    )

    (
        validation_loss,
        y_true,
        y_prob,
        y_pred
    ) = evaluate_mlp(
        mlp_model,
        validation_loader,
        criterion,
        device
    )

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Train loss: {train_loss:.4f} | "
        f"Validation loss: {validation_loss:.4f}"
    )

    history.append({
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'validation_loss':
            validation_loss
    })

Epoch 1/10 | Train loss: 1.0177 | Validation loss: 0.9984
Epoch 2/10 | Train loss: 0.9785 | Validation loss: 0.9805
Epoch 3/10 | Train loss: 0.9520 | Validation loss: 0.9702
Epoch 4/10 | Train loss: 0.9311 | Validation loss: 0.9670
Epoch 5/10 | Train loss: 0.9163 | Validation loss: 0.9712
Epoch 6/10 | Train loss: 0.9057 | Validation loss: 0.9772
Epoch 7/10 | Train loss: 0.9018 | Validation loss: 0.9828
Epoch 8/10 | Train loss: 0.8924 | Validation loss: 0.9905
Epoch 9/10 | Train loss: 0.8888 | Validation loss: 0.9967
Epoch 10/10 | Train loss: 0.8801 | Validation loss: 1.0035


In [25]:
# Training history
history_df = pd.DataFrame(
    history
)

display(
    history_df
)

,epoch,train_loss,validation_loss
0,1,1.017678,0.998387
1,2,0.978502,0.980490
2,3,0.951974,0.970167
3,4,0.931137,0.967027
4,5,0.916334,0.971204
5,6,0.905723,0.977176
6,7,0.901836,0.982803
7,8,0.892392,0.990475
8,9,0.888846,0.996658
9,10,0.880130,1.003513


In [26]:
# Structured baseline metrics
metrics = []

for i, label in enumerate(
    available_labels
):

    true_values = (
        y_true[:, i] == 1
    ).astype(int)

    probabilities = (
        y_prob[:, i]
    )

    predicted_values = (
        probabilities >= 0.5
    ).astype(int)

    accuracy = accuracy_score(
        true_values,
        predicted_values
    )

    precision = precision_score(
        true_values,
        predicted_values,
        zero_division=0
    )

    recall = recall_score(
        true_values,
        predicted_values,
        zero_division=0
    )

    f1 = f1_score(
        true_values,
        predicted_values,
        zero_division=0
    )

    try:

        roc_auc = roc_auc_score(
            true_values,
            probabilities
        )

    except ValueError:

        roc_auc = np.nan

    metrics.append({
        'label': label,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'roc_auc': roc_auc
    })

structured_baseline_metrics = pd.DataFrame(
    metrics
)

display(
    structured_baseline_metrics
)

,label,accuracy,precision,recall,f1,roc_auc
0,No Finding,0.656388,0.420455,0.578125,0.486842,0.701687
1,Support Devices,0.638767,0.530000,0.602273,0.563830,0.681409
2,Pleural Effusion,0.612335,0.375000,0.700000,0.488372,0.658483
3,Lung Opacity,0.541850,0.309091,0.548387,0.395349,0.586852
4,Atelectasis,0.563877,0.318182,0.593220,0.414201,0.587924
5,Cardiomegaly,0.621145,0.330097,0.666667,0.441558,0.680760
6,Edema,0.660793,0.223684,0.485714,0.306306,0.605506


In [27]:
# Overall metrics
y_true_binary = (
    y_true == 1
).astype(int)

y_pred_binary = (
    y_pred == 1
).astype(int)

overall_structured_metrics = pd.DataFrame({
    'metric': [
        'Accuracy',
        'Precision',
        'Recall',
        'F1',
        'ROC-AUC'
    ],

    'value': [

        accuracy_score(
            y_true_binary.flatten(),
            y_pred_binary.flatten()
        ),

        precision_score(
            y_true_binary.flatten(),
            y_pred_binary.flatten(),
            zero_division=0
        ),

        recall_score(
            y_true_binary.flatten(),
            y_pred_binary.flatten(),
            zero_division=0
        ),

        f1_score(
            y_true_binary.flatten(),
            y_pred_binary.flatten(),
            zero_division=0
        ),

        roc_auc_score(
            y_true_binary,
            y_prob,
            average='macro'
        )
    ]
})

display(
    overall_structured_metrics
)

,metric,value
0,Accuracy,0.613593
1,Precision,0.360515
2,Recall,0.601432
3,F1,0.450805
4,ROC-AUC,0.643232


In [28]:
# Save metrics
structured_metrics_file = (
    f'{processed_path}/'
    'structured_baseline_metrics.csv'
)

structured_baseline_metrics.to_csv(
    structured_metrics_file,
    index=False
)

print(
    "Saved:",
    structured_metrics_file
)

Saved: /content/drive/MyDrive/dissertation_project/data/processed/structured_baseline_metrics.csv


In [29]:
# Save training history
history_file = (
    f'{processed_path}/'
    'structured_training_history.csv'
)

history_df.to_csv(
    history_file,
    index=False
)

print(
    "Saved:",
    history_file
)

Saved: /content/drive/MyDrive/dissertation_project/data/processed/structured_training_history.csv


In [30]:
# Save scaler
import joblib

scaler_file = (
    f'{model_path}/'
    'structured_scaler.pkl'
)

joblib.dump(
    scaler,
    scaler_file
)

print(
    "Saved scaler:",
    scaler_file
)

Saved scaler: /content/drive/MyDrive/dissertation_project/data/models/structured_scaler.pkl


In [31]:
# Save
mlp_model_file = (
    f'{model_path}/'
    'structured_mlp.pt'
)

torch.save(
    {
        'model_state_dict':
            mlp_model.state_dict(),

        'input_features':
            feature_columns,

        'labels':
            available_labels,

        'feature_size':
            128
    },
    mlp_model_file
)

print(
    "Saved MLP:"
)

print(
    mlp_model_file
)

Saved MLP:
/content/drive/MyDrive/dissertation_project/data/models/structured_mlp.pt


In [32]:
# Feature extraction function
def extract_structured_features(
    model,
    X,
    device,
    batch_size=32
):

    tensor_data = torch.tensor(
        X,
        dtype=torch.float32
    )

    dataset = TensorDataset(
        tensor_data
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False
    )

    model.eval()

    features = []

    with torch.no_grad():

        for batch in loader:

            X_batch = (
                batch[0]
                .to(device)
            )

            _, feature_batch = model(
                X_batch
            )

            features.append(
                feature_batch
                .cpu()
                .numpy()
            )

    return np.vstack(
        features
    )

In [33]:
# Extract development features
development_structured_features = (
    extract_structured_features(
        mlp_model,
        X_development_scaled,
        device
    )
)

print(
    "Development structured feature shape:",
    development_structured_features.shape
)

Development structured feature shape: (1513, 128)


In [34]:
# Extract held-out features
heldout_structured_features = (
    extract_structured_features(
        mlp_model,
        X_heldout_scaled,
        device
    )
)

print(
    "Held-out structured feature shape:",
    heldout_structured_features.shape
)

Held-out structured feature shape: (687, 128)


In [35]:
# Save development structured features
structured_dev_df = pd.DataFrame(
    development_structured_features,
    columns=[
        f'structured_feature_{i}'
        for i in range(
            development_structured_features.shape[1]
        )
    ]
)

structured_dev_df.insert(
    0,
    'study_id',
    development_data[
        'study_id'
    ].values
)

structured_dev_file = (
    f'{processed_path}/'
    'structured_features_mlp.csv'
)

structured_dev_df.to_csv(
    structured_dev_file,
    index=False
)

print(
    "Saved:",
    structured_dev_file
)

Saved: /content/drive/MyDrive/dissertation_project/data/processed/structured_features_mlp.csv


In [36]:
# Save held-out structured features
structured_heldout_df = pd.DataFrame(
    heldout_structured_features,
    columns=[
        f'structured_feature_{i}'
        for i in range(
            heldout_structured_features.shape[1]
        )
    ]
)

structured_heldout_df.insert(
    0,
    'study_id',
    heldout_data[
        'study_id'
    ].values
)

structured_heldout_file = (
    f'{processed_path}/'
    'structured_features_mlp_heldout.csv'
)

structured_heldout_df.to_csv(
    structured_heldout_file,
    index=False
)

print(
    "Saved:",
    structured_heldout_file
)

Saved: /content/drive/MyDrive/dissertation_project/data/processed/structured_features_mlp_heldout.csv


In [37]:
import pandas as pd

processed_path = f'{base_path}/processed'

files_to_check = {
    'Structured (dev)':   'structured_features_mlp.csv',
    'Structured (heldout)': 'structured_features_mlp_heldout.csv',
    'Text (dev)':         'text_features_development.csv',
    'Text (heldout)':     'text_features_heldout.csv',
    'Image (dev)':        'image_features_development.csv',
    'Image (heldout)':    'image_features_heldout.csv',
    'Fusion (dev)':       'fusion_features_layer1.csv',
    'Fusion (heldout)':   'fusion_features_layer1_heldout.csv',
}

print(f"{'Modality':<22} {'Rows':>8} {'Columns':>10}")
print("-" * 42)

for label, filename in files_to_check.items():
    filepath = f'{processed_path}/{filename}'
    try:
        df = pd.read_csv(filepath)
        print(f"{label:<22} {df.shape[0]:>8} {df.shape[1]:>10}")
    except FileNotFoundError:
        print(f"{label:<22} {'MISSING':>8}")

Modality                   Rows    Columns
------------------------------------------
Structured (dev)           1513        129
Structured (heldout)        687        129
Text (dev)                 1513        769
Text (heldout)              687        769
Image (dev)                1513        769
Image (heldout)             687        769
Fusion (dev)               1513        129
Fusion (heldout)            687        129
